# Tempo de sentença nos Juizados Especiais

Neste notebook baixei a base Justiça em Números e separei o tempo médio da sentença na fase de conhecimento dos Juizados Especiais dos 27 Tribunais de Justiça.

In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
import requests

#projeto feito com o auxilio de IA Chat GPT
PASTA_PROJETO = Path.cwd()
if PASTA_PROJETO.name.lower() == "src":
    PASTA_PROJETO = PASTA_PROJETO.parent
if not (PASTA_PROJETO / "data").is_dir():
    raise FileNotFoundError("Execute o notebook pela raiz do projeto ou pela pasta src.")
ARQUIVO_BRUTO = PASTA_PROJETO / "data" / "raw" / "justica_em_numeros_2026.zip"
ARQUIVO_TRATADO = PASTA_PROJETO / "data" / "processed" / "tempo_sentenca_juizados.csv"

In [ ]:
# Download da base publicada pelo CNJ em 23 de junho de 2026.
resposta = requests.get(
    "https://www.cnj.jus.br/wp-content/uploads/2026/06/23-jun-2026.zip",
    headers={"User-Agent": "Mozilla/5.0 (projeto acadêmico TP2)"},
    timeout=180,
)
print("Status do download:", resposta.status_code)
resposta.raise_for_status()
ARQUIVO_BRUTO.write_bytes(resposta.content)

In [ ]:
# Leitura do arquivo principal que estava dentro do ZIP.
with zipfile.ZipFile(ARQUIVO_BRUTO) as arquivo_zip:
    print("Arquivos encontrados:", arquivo_zip.namelist())
    dados = pd.read_csv(
        arquivo_zip.open("JN_23-Jun-2026.csv"),
        sep=";",
        encoding="cp1252",
        decimal=",",
        na_values=["nd", "n/a"],
        low_memory=False,
    )

print("Linhas e colunas da base:", dados.shape)
dados[["ano", "justica", "sigla", "tpsentcjem"]].tail()

In [ ]:
# Recorte da Justiça Estadual no ano de 2025.
tempo_sentenca = dados.loc[
    (dados["ano"] == 2025)
    & (dados["justica"] == "Estadual")
    & (dados["sigla"] != "TJ"),
    ["ano", "sigla", "dsc_tribunal", "uf_sede", "tpsentcjem", "sentcje"],
].copy()

tempo_sentenca.columns = [
    "ano",
    "tribunal",
    "nome_tribunal",
    "uf",
    "tempo_medio_sentenca_dias",
    "sentencas_conhecimento",
]
tempo_sentenca["tempo_medio_sentenca_dias"] = tempo_sentenca["tempo_medio_sentenca_dias"].round(1)
tempo_sentenca = tempo_sentenca.sort_values("tempo_medio_sentenca_dias").reset_index(drop=True)
tempo_sentenca

In [ ]:
# Testes feitos antes de salvar o resultado.
assert len(tempo_sentenca) == 27
assert tempo_sentenca["tribunal"].nunique() == 27
assert tempo_sentenca["tempo_medio_sentenca_dias"].notna().all()
assert (tempo_sentenca["tempo_medio_sentenca_dias"] > 0).all()

print("Testes concluídos sem erros.")
print("Menor tempo:", tempo_sentenca.iloc[0]["tribunal"], tempo_sentenca.iloc[0]["tempo_medio_sentenca_dias"])
print("Maior tempo:", tempo_sentenca.iloc[-1]["tribunal"], tempo_sentenca.iloc[-1]["tempo_medio_sentenca_dias"])

In [ ]:
tempo_sentenca.to_csv(ARQUIVO_TRATADO, index=False, encoding="utf-8-sig")
print("Arquivo bruto:", ARQUIVO_BRUTO)
print("Arquivo tratado:", ARQUIVO_TRATADO)